In [ ]:
tabella["standard_value"] = pd.to_numeric(
    tabella["standard_value"],
    errors="coerce"
)

print(tabella["standard_value"].dtype)

float64


In [ ]:
print(tabella["standard_value"].head(10))

0         41.0
1        300.0
2       7820.0
3        170.0
4         40.0
5        440.0
6       9300.0
7     500000.0
8    3000000.0
9      96000.0
Name: standard_value, dtype: object


In [ ]:
print(tabella["standard_units"].value_counts(dropna=False))

standard_units
nM    10
Name: count, dtype: int64


In [ ]:
attivita = data["activities"]

tabella = pd.DataFrame(attivita)

tabella[
    [
        "molecule_chembl_id",
        "standard_type",
        "standard_value",
        "standard_units",
        "assay_chembl_id"
    ]
]

,molecule_chembl_id,standard_type,standard_value,standard_units,assay_chembl_id
0,CHEMBL68920,IC50,41.0,nM,CHEMBL674637
1,CHEMBL68920,IC50,300.0,nM,CHEMBL621151
2,CHEMBL68920,IC50,7820.0,nM,CHEMBL615325
3,CHEMBL69960,IC50,170.0,nM,CHEMBL674637
4,CHEMBL69960,IC50,40.0,nM,CHEMBL621151
5,CHEMBL69960,IC50,440.0,nM,CHEMBL615325
6,CHEMBL137635,IC50,9300.0,nM,CHEMBL677833
7,CHEMBL306988,IC50,500000.0,nM,CHEMBL674643
8,CHEMBL66879,IC50,3000000.0,nM,CHEMBL674643
9,CHEMBL77085,IC50,96000.0,nM,CHEMBL674643


In [ ]:
prima_attivita = data["activities"][0]

print(prima_attivita)

{'action_type': None, 'activity_comment': None, 'activity_id': 32260, 'activity_properties': [], 'assay_chembl_id': 'CHEMBL674637', 'assay_description': 'Inhibitory activity towards tyrosine phosphorylation for the epidermal growth factor-receptor kinase', 'assay_type': 'B', 'assay_variant_accession': None, 'assay_variant_mutation': None, 'bao_endpoint': 'BAO_0000190', 'bao_format': 'BAO_0000357', 'bao_label': 'single protein format', 'canonical_smiles': 'Cc1cc(C)c(/C=C2\\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)c32)[nH]1', 'data_validity_comment': None, 'data_validity_description': None, 'document_chembl_id': 'CHEMBL1134862', 'document_journal': 'Bioorg Med Chem Lett', 'document_year': 2002, 'ligand_efficiency': {'bei': '19.25', 'le': '0.37', 'lle': '2.94', 'sei': '8.93'}, 'modality': None, 'molecule_chembl_id': 'CHEMBL68920', 'molecule_pref_name': None, 'parent_molecule_chembl_id': 'CHEMBL68920', 'pchembl_value': '7.39', 'potential_duplicate': 0, 'qudt_units': 'http://www.openphacts.org/units/N

In [ ]:
url = "https://www.ebi.ac.uk/chembl/api/data/activity.json"

params = {
    "target_chembl_id": "CHEMBL203",
    "standard_type": "IC50",
    "limit": 10
}

response = requests.get(url, params=params)

print(response.status_code)

data = response.json()

print("Numero di attività:", len(data["activities"]))

200
Numero di attività: 10


In [ ]:
data = response.json()

print(data.keys())

dict_keys(['activities', 'page_meta'])


In [ ]:
import requests
import pandas as pd

url = "https://www.ebi.ac.uk/chembl/api/data/activity.json"

params = {
    "standard_type": "IC50",
    "limit": 10
}

response = requests.get(url, params=params)

print(response.status_code)

200


In [ ]:
ranking = risultati.sort_values(
    "AL_score",
    ascending=False
)

ranking[
    [
        "molecule_id",
        "Predizione_pIC50",
        "Incertezza",
        "Diversita",
        "ApplicabilityDomain",
        "AL_score"
    ]
]

,molecule_id,Predizione_pIC50,Incertezza,Diversita,ApplicabilityDomain,AL_score
4,M5,7.7815,0.064528,0.555556,frontier,0.6
3,M4,7.5896,0.092483,0.444444,inside-domain,0.4


In [ ]:
risultati["AL_score"] = (
    0.40 * risultati["Incertezza_norm"]
    + 0.40 * risultati["Diversita_norm"]
    + 0.20 * risultati["Predizione_norm"]
)

risultati

,molecule_id,SMILES,Predizione_pIC50,Incertezza,SimilaritaMassima,Diversita,ApplicabilityDomain,Predizione_norm,Incertezza_norm,Diversita_norm,AL_score
3,M4,CCCN,7.5896,0.092483,0.555556,0.444444,inside-domain,0.0,1.0,0.0,0.4
4,M5,CCCC,7.7815,0.064528,0.444444,0.555556,frontier,1.0,0.0,1.0,0.6


In [ ]:
def normalizza(colonna):
    minimo = colonna.min()
    massimo = colonna.max()

    if massimo == minimo:
        return colonna * 0

    return (colonna - minimo) / (massimo - minimo)


risultati["Predizione_norm"] = normalizza(
    risultati["Predizione_pIC50"]
)

risultati["Incertezza_norm"] = normalizza(
    risultati["Incertezza"]
)

risultati["Diversita_norm"] = normalizza(
    risultati["Diversita"]
)

risultati

,molecule_id,SMILES,Predizione_pIC50,Incertezza,SimilaritaMassima,Diversita,ApplicabilityDomain,Predizione_norm,Incertezza_norm,Diversita_norm
3,M4,CCCN,7.5896,0.092483,0.555556,0.444444,inside-domain,0.0,1.0,0.0
4,M5,CCCC,7.7815,0.064528,0.444444,0.555556,frontier,1.0,0.0,1.0


In [ ]:
risultati["ApplicabilityDomain"] = risultati["SimilaritaMassima"].apply(
    classifica_AD
)

risultati

,molecule_id,SMILES,Predizione_pIC50,Incertezza,SimilaritaMassima,Diversita,ApplicabilityDomain
3,M4,CCCN,7.5896,0.092483,0.555556,0.444444,inside-domain
4,M5,CCCC,7.7815,0.064528,0.444444,0.555556,frontier


In [ ]:
def classifica_AD(similarita):
    if similarita >= 0.5:
        return "inside-domain"
    elif similarita >= 0.3:
        return "frontier"
    else:
        return "outside-domain"

In [ ]:
risultati["SimilaritaMassima"] = candidates.apply(
    lambda r: similarita_massima(r, known),
    axis=1
)

risultati["Diversita"] = 1 - risultati["SimilaritaMassima"]

risultati

,molecule_id,SMILES,Predizione_pIC50,Incertezza,SimilaritaMassima,Diversita
3,M4,CCCN,7.5896,0.092483,0.555556,0.444444
4,M5,CCCC,7.7815,0.064528,0.444444,0.555556


In [ ]:
def similarita_massima(candidata, conosciute):
    similarita = []

    for _, molecola in conosciute.iterrows():
        sim = TanimotoSimilarity(
            candidata["Fingerprint"],
            molecola["Fingerprint"]
        )
        similarita.append(sim)

    return max(similarita)

In [ ]:
known = df[df["pIC50"].notna()]
candidates = df[df["pIC50"].isna()]

for _, candidata in candidates.iterrows():
    print("Candidata:", candidata["molecule_id"])

    for _, conosciuta in known.iterrows():
        similarita = TanimotoSimilarity(
            candidata["Fingerprint"],
            conosciuta["Fingerprint"]
        )

        print(
            "  rispetto a",
            conosciuta["molecule_id"],
            ":",
            round(similarita, 3)
        )

Candidata: M4
  rispetto a M1 : 0.273
  rispetto a M2 : 0.556
  rispetto a M3 : 0.333
Candidata: M5
  rispetto a M1 : 0.375
  rispetto a M2 : 0.375
  rispetto a M3 : 0.444


In [ ]:
from rdkit.DataStructs import TanimotoSimilarity

fp_M1 = df.loc[df["molecule_id"] == "M1", "Fingerprint"].iloc[0]
fp_M2 = df.loc[df["molecule_id"] == "M2", "Fingerprint"].iloc[0]

similarita = TanimotoSimilarity(fp_M1, fp_M2)

print("Similarità M1-M2:", similarita)

Similarità M1-M2: 0.3333333333333333


In [ ]:
from rdkit.Chem import AllChem

def crea_fingerprint(smiles):
    molecola = Chem.MolFromSmiles(smiles)
    return AllChem.GetMorganFingerprintAsBitVect(
        molecola,
        radius=2,
        nBits=2048
    )

df["Fingerprint"] = df["SMILES"].apply(crea_fingerprint)

In [ ]:
risultati = candidates[["molecule_id", "SMILES"]].copy()

risultati["Predizione_pIC50"] = media_predizioni
risultati["Incertezza"] = incertezza

risultati

,molecule_id,SMILES,Predizione_pIC50,Incertezza
3,M4,CCCN,7.5896,0.092483
4,M5,CCCC,7.7815,0.064528


In [ ]:
incertezza = predizioni_ensemble.std(axis=0)

print("Incertezza:")
print(incertezza)

Incertezza:
[0.09248265 0.0645279 ]


In [ ]:
media_predizioni = predizioni_ensemble.mean(axis=0)

print("Predizione media:")
print(media_predizioni)

Predizione media:
[7.5896 7.7815]


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor

predizioni_ensemble = []

for i in range(10):
    modello = RandomForestRegressor(
        n_estimators=100,
        random_state=i
    )

    modello.fit(X_train, y_train)

    pred = modello.predict(X_candidates)
    predizioni_ensemble.append(pred)

predizioni_ensemble = np.array(predizioni_ensemble)

print(predizioni_ensemble)

[[7.608 7.733]
 [7.688 7.843]
 [7.546 7.764]
 [7.619 7.861]
 [7.519 7.723]
 [7.524 7.73 ]
 [7.729 7.902]
 [7.393 7.692]
 [7.61  7.772]
 [7.66  7.795]]


In [ ]:
predictions = model.predict(X_candidates)

print(predictions)

[7.636 7.752]


In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [ ]:
X_train = training[features]
y_train = training["pIC50"]

X_candidates = candidates[features]

In [ ]:
features = [
    "MW",
    "logP",
    "TPSA",
    "HBD",
    "HBA",
    "RotatableBonds",
    "RingCount"
]

In [ ]:
candidates = df[df["pIC50"].isna()]

print(candidates)

  molecule_id SMILES  pIC50  SMILES_validi      MW    logP   TPSA  HBD  HBA  \
3          M4   CCCN    NaN           True  59.112  0.3551  26.02    1    1   
4          M5   CCCC    NaN           True  58.124  1.8064   0.00    0    0   

   RotatableBonds  RingCount  
3               1          0  
4               1          0  


In [ ]:
training = df[df["pIC50"].notna()]

print(training)

  molecule_id SMILES  pIC50  SMILES_validi      MW    logP   TPSA  HBD  HBA  \
0          M1    CCO    7.2           True  46.069 -0.0014  20.23    1    1   
1          M2    CCN    6.5           True  45.085 -0.0350  26.02    1    1   
2          M3   CCCO    8.1           True  60.096  0.3887  20.23    1    1   

   RotatableBonds  RingCount  
0               0          0  
1               0          0  
2               1          0  


In [ ]:
from rdkit.Chem import AllChem

smiles = "CCO"
molecola = Chem.MolFromSmiles(smiles)

fingerprint = AllChem.GetMorganFingerprintAsBitVect(
    molecola,
    radius=2,
    nBits=2048
)

print(fingerprint)

In [ ]:
def calcola_rotatable_bonds(smiles):
    molecola = Chem.MolFromSmiles(smiles)
    return Descriptors.NumRotatableBonds(molecola)

def calcola_ring_count(smiles):
    molecola = Chem.MolFromSmiles(smiles)
    return Descriptors.RingCount(molecola)

df["RotatableBonds"] = df["SMILES"].apply(calcola_rotatable_bonds)
df["RingCount"] = df["SMILES"].apply(calcola_ring_count)

df

,molecule_id,SMILES,pIC50,SMILES_validi,MW,logP,TPSA,HBD,HBA,RotatableBonds,RingCount
0,M1,CCO,7.2,True,46.069,-0.0014,20.23,1,1,0,0
1,M2,CCN,6.5,True,45.085,-0.0350,26.02,1,1,0,0
2,M3,CCCO,8.1,True,60.096,0.3887,20.23,1,1,1,0
3,M4,CCCN,NaN,True,59.112,0.3551,26.02,1,1,1,0
4,M5,CCCC,NaN,True,58.124,1.8064,0.00,0,0,1,0


In [ ]:
def calcola_HBD(smiles):
    molecola = Chem.MolFromSmiles(smiles)
    return Descriptors.NumHDonors(molecola)

def calcola_HBA(smiles):
    molecola = Chem.MolFromSmiles(smiles)
    return Descriptors.NumHAcceptors(molecola)

df["HBD"] = df["SMILES"].apply(calcola_HBD)
df["HBA"] = df["SMILES"].apply(calcola_HBA)

df

,molecule_id,SMILES,pIC50,SMILES_validi,MW,logP,TPSA,HBD,HBA
0,M1,CCO,7.2,True,46.069,-0.0014,20.23,1,1
1,M2,CCN,6.5,True,45.085,-0.0350,26.02,1,1
2,M3,CCCO,8.1,True,60.096,0.3887,20.23,1,1
3,M4,CCCN,NaN,True,59.112,0.3551,26.02,1,1
4,M5,CCCC,NaN,True,58.124,1.8064,0.00,0,0


In [ ]:
def calcola_TPSA(smiles):
    molecola = Chem.MolFromSmiles(smiles)
    return Descriptors.TPSA(molecola)

df["TPSA"] = df["SMILES"].apply(calcola_TPSA)

df

,molecule_id,SMILES,pIC50,SMILES_validi,MW,logP,TPSA
0,M1,CCO,7.2,True,46.069,-0.0014,20.23
1,M2,CCN,6.5,True,45.085,-0.0350,26.02
2,M3,CCCO,8.1,True,60.096,0.3887,20.23
3,M4,CCCN,NaN,True,59.112,0.3551,26.02
4,M5,CCCC,NaN,True,58.124,1.8064,0.00


In [ ]:
def calcola_logP(smiles):
    molecola = Chem.MolFromSmiles(smiles)
    return Descriptors.MolLogP(molecola)

df["logP"] = df["SMILES"].apply(calcola_logP)

df

,molecule_id,SMILES,pIC50,SMILES_validi,MW,logP
0,M1,CCO,7.2,True,46.069,-0.0014
1,M2,CCN,6.5,True,45.085,-0.0350
2,M3,CCCO,8.1,True,60.096,0.3887
3,M4,CCCN,NaN,True,59.112,0.3551
4,M5,CCCC,NaN,True,58.124,1.8064


In [ ]:
def calcola_MW(smiles):
    molecola = Chem.MolFromSmiles(smiles)
    return Descriptors.MolWt(molecola)

df["MW"] = df["SMILES"].apply(calcola_MW)

df

,molecule_id,SMILES,pIC50,SMILES_validi,MW
0,M1,CCO,7.2,True,46.069
1,M2,CCN,6.5,True,45.085
2,M3,CCCO,8.1,True,60.096
3,M4,CCCN,NaN,True,59.112
4,M5,CCCC,NaN,True,58.124


In [ ]:
from rdkit.Chem import Descriptors

smiles = "CCO"

molecola = Chem.MolFromSmiles(smiles)

peso_molecolare = Descriptors.MolWt(molecola)

print("Peso molecolare:", peso_molecolare)

Peso molecolare: 46.069


In [ ]:
df["SMILES_validi"] = df["SMILES"].apply(controlla_smiles)

df

,molecule_id,SMILES,pIC50,SMILES_validi
0,M1,CCO,7.2,True
1,M2,CCN,6.5,True
2,M3,CCCO,8.1,True
3,M4,CCCN,NaN,True
4,M5,CCCC,NaN,True


In [ ]:
def controlla_smiles(smiles):
    molecola = Chem.MolFromSmiles(smiles)

    if molecola is None:
        return False
    else:
        return True

In [ ]:
smiles = "CCO"

molecola = Chem.MolFromSmiles(smiles)

print(molecola)

In [ ]:
from rdkit import Chem
print("RDkit funziona!")

RDkit funziona!


In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 40.1 MB/s eta 0:00:00


In [ ]:
print("Numero totale di molecole:", len(df))
print("Numero di candidate:", len(candidati))


Numero totale di molecole: 5
Numero di candidate: 2


In [ ]:
candidati = df[df["pIC50"].isna()]

print(candidati)

  molecule_id SMILES  pIC50
3          M4   CCCN    NaN
4          M5   CCCC    NaN


In [ ]:
df

,molecule_id,SMILES,pIC50
0,M1,CCO,7.2
1,M2,CCN,6.5
2,M3,CCCO,8.1
3,M4,CCCN,NaN
4,M5,CCCC,NaN


In [ ]:
import pandas as pd
dati = {
    "molecule_id": ["M1", "M2", "M3", "M4", "M5"],
    "SMILES": ["CCO", "CCN", "CCCO", "CCCN", "CCCC"],
    "pIC50": [7.2, 6.5, 8.1, None, None]
}

df = pd.DataFrame(dati)

print(df)

  molecule_id SMILES  pIC50
0          M1    CCO    7.2
1          M2    CCN    6.5
2          M3   CCCO    8.1
3          M4   CCCN    NaN
4          M5   CCCC    NaN


In [ ]:
# (ciao!sto iniziando il progetto di active learning-ED)
print("Activelearn-ED")
print("AI for Drug discovery")

Activelearn-ED
AI for Drug discovery
